# train.ipynb — 1 enc 3 dec 통합 학습 (담당: 세원)

1. 환경 준비 → 2. 설정 → 3. 데이터 → 4. 모델 → 5. 학습 → 6. 방향별 평가 → 7. 6개 파일 저장(Drive) → 8. 저장 파일 확인

`TODO(…)` 표시는 다른 파트 코드가 push되면 함수 이름·인자를 맞출 곳이다.

In [ ]:
# 1. 환경 준비 (Colab / 로컬 둘 다 동작)
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists("/content/2026-nlp"):
        !git -C /content/2026-nlp pull
    else:
        !git clone https://github.com/sloth2003/2026-nlp.git /content/2026-nlp
    REPO = "/content/2026-nlp"
    CKPT_DIR = "/content/drive/MyDrive/2026-nlp/checkpoints"   # 학습 중간 저장 (세션 끊김 대비)
else:
    REPO = os.path.abspath("../..")                              # notebook/ 기준
    CKPT_DIR = os.path.join(REPO, "project01-dev", "train")

DEV = os.path.join(REPO, "project01-dev")
SUBMIT_DIR = os.path.join(CKPT_DIR, "final")                    # 제출용 .pt 6개 (여기서 내려받아 사람이 project01/ 에 올린다)
os.makedirs(CKPT_DIR, exist_ok=True)
sys.path.insert(0, os.path.join(DEV, "src"))
print(REPO, CKPT_DIR, SUBMIT_DIR, sep="\n")

In [ ]:
# 2. 설정
import json, random
import numpy as np
import torch

with open(os.path.join(DEV, "config", "split.json"), encoding="utf-8") as f:
    config = json.load(f)

SEED = config["random_seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| config:", config)

MODEL_KWARGS = dict(embedding_dim=128, hidden_dim=256, num_layers=2, dropout=0.1)   # model.py 인자와 같게

In [ ]:
# 3. 데이터
# data.py(재후)의 combined, Vocab, NumDataset, collate_fn, pair_df 를 그대로 쓰고,
# 1 enc 3 dec 에 필요한 세 가지만 여기서 만든다.
#   (1) 숫자 기준 split 한 번 -> 6방향이 같은 train / valid / test 를 쓴다
#   (2) 입력 vocab 통합 -> 세 언어 토큰을 합친 사전 하나 (id 공유), 출력 vocab 은 언어별
#   (3) 6방향 loader 묶음 -> loaders["kor2eng"]["train" / "valid" / "test"]
import data
from torch.utils.data import DataLoader, Subset
from seq2seq import DIRECTIONS, LANGS

combined = data.combined                     # num, kor, eng (한쪽 파일에만 있는 숫자는 NaN)

# (1) 숫자 기준 split
nums = sorted(combined["num"])               # 문자열 그대로 정렬 (int 변환 없음)
g = torch.Generator().manual_seed(config["random_seed"])
order = torch.randperm(len(nums), generator=g).tolist()
n_train = int(len(nums) * config["train_ratio"])
n_valid = int(len(nums) * config["valid_ratio"])
split_of = {}
for rank, i in enumerate(order):
    split_of[nums[i]] = "train" if rank < n_train else "valid" if rank < n_train + n_valid else "test"

# (2) vocab
tgt_vocabs = {lang: data.Vocab(combined[lang].dropna(), lang) for lang in LANGS}
joint_tokens = sorted({t for v in tgt_vocabs.values() for t in v.itos[len(data.SPECIALS):]})

def make_src_vocab(lang):
    """통합 사전(id 공유) + 입력 언어의 토큰화 방식(col)"""
    v = data.Vocab([], lang)
    v.itos = data.SPECIALS + joint_tokens
    v.stoi = {t: i for i, t in enumerate(v.itos)}
    return v

src_vocabs = {lang: make_src_vocab(lang) for lang in LANGS}   # 세 개 모두 id 가 같고 col 만 다르다
src_vocab = src_vocabs["num"]                                 # 모델 크기·저장용 대표 (checkpoint 가 방향별로 col 을 바꿔 저장)

# (3) 6방향 loader
loaders = {}
for d in DIRECTIONS:
    s, t = d.split("2")
    df = data.pair_df(combined, s, t)                         # 두 열이 다 있는 행만 (kor-eng 는 공통 숫자만)
    ds = data.NumDataset(df, s, t, src_vocabs[s], tgt_vocabs[t])
    loaders[d] = {}
    for part in ("train", "valid", "test"):
        idx = [i for i, n in enumerate(df["num"]) if split_of[n] == part]
        loaders[d][part] = DataLoader(Subset(ds, idx), batch_size=config["batch_size"],
                                      shuffle=(part == "train"), collate_fn=data.collate_fn)

print("입력 vocab:", len(src_vocab), "| 출력 vocab:", {l: len(v) for l, v in tgt_vocabs.items()})
for d in DIRECTIONS:
    print(f"{d:8s}", {k: len(loaders[d][k].dataset) for k in ("train", "valid", "test")})

In [ ]:
# 4. 모델: 공유 Encoder 1개 + Decoder 3개
from seq2seq import build_model

model = build_model(len(src_vocab), {l: len(v) for l, v in tgt_vocabs.items()}, **MODEL_KWARGS).to(device)
print(model)
print("파라미터:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 5. 학습
# 셀을 실행할 때마다 마지막으로 저장된 지점부터 EPOCHS_PER_RUN 만큼 더 학습하고 last.pt 에 저장한다.
# 처음부터 다시 학습하려면 Drive 의 last.pt 를 지운다 (모델 크기·vocab 을 바꿨을 때도 반드시 지울 것).
#
# TODO(성현 train.py): 함수 이름·인자 맞추기. 안에서 할 일:
#   start_epoch 부터 end_epoch - 1 까지 반복
#   6방향 배치를 비율대로 섞어서 logits = model(x, y, tgt_lang), loss = batch_loss(y[:, 1:], logits[:, :-1])
#   epoch 마다 train loss, valid loss, 방향별 valid 정확도를 한 줄로 출력
#   epoch 마다 save_training_state(ckpt_path, model, optimizer, epoch) 로 중간 저장 (세션이 끊겨도 그 epoch 까지 남게)
from train import train
from checkpoint import save_training_state, load_training_state

EPOCHS_PER_RUN = 5                                                # 이 셀 한 번 실행에 돌릴 epoch 수
LAST = os.path.join(CKPT_DIR, "last.pt")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
start_epoch, best = 0, None
if os.path.exists(LAST):                                          # 저장된 지점이 있으면 모델·optimizer 를 거기로 되돌림
    last_epoch, best = load_training_state(LAST, model, optimizer, device)
    start_epoch = last_epoch + 1
    print(f"[이어서 학습] {LAST} 에서 불러옴 -> epoch {start_epoch}부터 (저장된 마지막 epoch {last_epoch})")
else:
    print(f"[처음부터 학습] {LAST} 없음 -> epoch 0부터")
end_epoch = start_epoch + EPOCHS_PER_RUN
print(f"이번 실행: epoch {start_epoch} ~ {end_epoch - 1} ({EPOCHS_PER_RUN} epoch)")

history = train(model, loaders, optimizer, config, device=device,  # TODO(성현)
                start_epoch=start_epoch, end_epoch=end_epoch, ckpt_path=LAST)

# 셀이 끝날 때 한 번 더 저장 (train() 안에서 저장하지 않아도 다음 실행 때 이어서 학습되도록)
save_training_state(LAST, model, optimizer, epoch=end_epoch - 1, best_score=best)
print(f"epoch {start_epoch}~{end_epoch - 1} 학습 완료 -> 저장: {LAST}")

In [ ]:
# 6. 방향별 test 평가 (beam)
from inference import evaluate

model.eval()
results = {}
for d in DIRECTIONS:
    s, t = d.split("2")
    r = evaluate(model.direction(d), loaders[d]["test"], tgt_vocabs[t],
                 method="beam", beam_width=5, src_vocab=src_vocabs[s])
    results[d] = r["acc"]
    print(f"{d:8s} acc {r['acc']:.4f} ({r['correct']}/{r['total']})  틀린 예: {r['errors'][:2]}")

In [ ]:
# 7. 제출용 .pt 6개 저장 (Drive: MyDrive/2026-nlp/checkpoints/final/)
# GitHub 에는 올리지 않는다. Drive 에서 6개를 내려받아 사람이 project01/ 에 commit 한다.
from checkpoint import save_all

paths = save_all(SUBMIT_DIR, model, src_vocab, tgt_vocabs, MODEL_KWARGS)
for p in paths:
    print(p, f"{os.path.getsize(p) / 1e6:.1f} MB")

In [ ]:
# 8. 저장한 파일(Drive final/)만으로 다시 로드해서 번역 확인
from checkpoint import load_model
from inference import translate

examples = {"num": "12708", "kor": "만 이천칠백팔", "eng": "twelve thousand seven hundred eight"}
for d in DIRECTIONS:
    src, _ = d.split("2")
    m, sv, tv = load_model(os.path.join(SUBMIT_DIR, f"{d}.pt"), device=device)
    print(f"{d:8s} {examples[src]!r} -> {translate(m, examples[src], sv, tv, method='beam')!r}")